# Langchain 

In [55]:
!pip install -qU langchain "langchain[openai]"
!pip install requests
!pip install python-dotenv
!pip install langchain-community
!pip install langchain-community faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.3 MB 4.4 MB/s eta 0:00:04
   ---- ----------------------------------- 1.8/16.3 MB 3.9 MB/s eta 0:00:04
   ------- -------------------------------- 2.9/16.3 MB 4.2 MB/s eta 0:00:04
   --------- ------------------------------ 3.9/16.3 MB 4.4 MB/s eta 0:00:03
   ------------ --------------------------- 5.0/16.3 MB 4.5 MB/s eta 0:00:03
   -------------- ------------------------- 6.0/16.3 MB 4.6 MB/s eta 0:00:03
   ----------------- ---------------------- 7.1/16.3 MB 4.7 MB/s eta 0:00:02
   -------------------- ------------------- 8.4/16.3 MB 4.8 MB/s eta 0:00:02
   ----------------------- ---------------- 9.4/16.3 MB 4.9 MB/s eta 0:00:02
   -------------------------- ------------- 10.7/16.3 MB 4.9 MB/s eta 0:00:02
   --------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [40]:
import requests
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool,ToolRuntime
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from dataclasses import dataclass
load_dotenv()
# load_dotenv(Path(__file__).with_name(".env"))


True

In [28]:
@tool('get_current_weather', description="Get the current weather for a given location.")
def get_current_weather(location: str) -> str:
    return requests.get(f"https://wttr.in/{location}?format=j1").json()

In [33]:
# 1. On initialise le modèle
model = init_chat_model("gpt-4.1-mini", model_provider="openai",temperature=0.5)

# 2. On crée l'agent en associant le modèle et les outils
agent = create_agent(
    tools=[get_current_weather],
    model=model,
    system_prompt="You are a helpful assistant that provides weather information.",
)
#3. On envoie un message simple
response = agent.invoke({
    'messages':[
        {"role": "user", "content": "What is the current weather in Paris?"}
    ]
})

print (response['messages'][-1].content)
#4. On envoie conversation complète

conversation =[
    SystemMessage(content="You are a helpful assistant that provides weather information."),
    HumanMessage(content="What is the current weather in Paris?"),
    AIMessage(content="The current weather in Paris is sunny with a temperature of 32°C (90°F)"),
    HumanMessage(content="What is the current weather in London?")
]
response = model.invoke(conversation)
print (response.content)

The current weather in Paris is partly cloudy with a temperature of 32°C (90°F). The humidity is at 32%, and the wind is coming from the west at 25 km/h (15 mph). There is no precipitation at the moment. The visibility is 10 km (6 miles).
I’m unable to provide real-time weather updates. For the current weather in London, please check a reliable weather website or app like Weather.com or AccuWeather.


- L'objetcif est de renvoyer la temperature en fonction de la localisation de l utilisateur qui est connue.
- On souhaite gealement structurer cette reponse 

In [48]:
@dataclass
class Context:
    user_id: str

@dataclass 
class ResponseFormat:
    summary: str
    temperature: float
    humidity: float

# On cree un tool locate_user qui prend en parametre un ToolRuntime[Context] qui fait reference a Context, qui contient context.user_id, retourne la localisation de l'utilisateur en fonction de son user_id.
@tool('locate_user', description="Get user localization based on user_id.")
def locate_user(runtime:ToolRuntime[Context]) -> str:
    match runtime.context.user_id:
        case "user_1":
            return "Paris"
        case "user_2":
            return "London"
        case _:
            return "Unknown location"

checkpointer = InMemorySaver()

agent2 = create_agent(
    tools=[get_current_weather,locate_user],
    model=model,
    system_prompt="You are a helpful assistant that provides weather information.",
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

config = {'configurable':{'thread_id': 1}}
response = agent2.invoke({
    'messages':[
        {"role": "user", "content": "What is the current weather?"}
    ]},
    context=Context(user_id="user_1"),
    config=config
)

print(response['structured_response'])
print(response['structured_response'].summary)
print(response['structured_response'].temperature)
print(response['structured_response'].humidity)

response = agent2.invoke({
    'messages':[
        {"role": "user", "content": "What is the current weather?"}
    ]},
    context=Context(user_id="user_2"),
    config=config
)

print(response['structured_response'])
print(response['structured_response'].summary)
print(response['structured_response'].temperature)
print(response['structured_response'].humidity)

C:\Users\pierr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(
Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'ResponseFormat')]


ResponseFormat(summary='The current weather in Paris is partly cloudy with a temperature of 30°C. The humidity is at 37%.', temperature=30.0, humidity=37.0)
The current weather in Paris is partly cloudy with a temperature of 30°C. The humidity is at 37%.
30.0
37.0


C:\Users\pierr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


ResponseFormat(summary='The current weather in London is sunny with a temperature of 26°C. The humidity is at 46%.', temperature=26.0, humidity=46.0)
The current weather in London is sunny with a temperature of 26°C. The humidity is at 46%.
26.0
46.0


#### Petit Rag

In [58]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)
vectorstore = FAISS.from_texts(
    texts=["It's sunny in ville d'avray", "It's rainy in versailles","I don't know the weather in Paris"], 
    embedding=embeddings
)
print(vectorstore.similarity_search("It's sunny in ville d'avray", k=3))

[Document(id='4ae2a703-6da9-420a-9c90-54bf339cf86a', metadata={}, page_content="It's sunny in ville d'avray"), Document(id='8c2a0428-af29-4bbb-a54e-d0f881478e66', metadata={}, page_content="It's rainy in versailles"), Document(id='7d83db4c-81f5-4c04-b798-1c62c4296666', metadata={}, page_content="I don't know the weather in Paris")]


In [ ]:
retierver = vectorstore.as_retriever(search_kwargs={"k": 1})
retriever_tool = create_retriever_tool(retriever,
    name="faiss_retriever",
    description="A tool to retrieve documents from a FAISS vector store.")

agent3 = create_agent(
    tools=[retriever_tool],
    model=model,
    system_prompt="You are a helpful assistant that retrieves documents from a FAISS vector store. For questions about the weather, you can use the retriever_tool tool to provide accurate information.Maybe you have to do it multiple times before answering the question.",
)

response = agent3.invoke({
    'messages':[
        {"role": "user", "content": "What's the weather like in ville d'avray?"}
    ]
})

print (response['messages'][-1].content)


TypeError: create_retriever_tool() missing 1 required positional argument: 'retriever'